![Pythonista](img/pythonista.png)

# *Pipelines* de infraestructura: *plan*, *apply* y estrategia de *runners*.

Esta sesión conecta la disciplina de *Terraform* con una estrategia operativa concreta: *hosted* por defecto y *self-hosted* solo cuando existe una necesidad técnica o regulatoria verificable.

## Objetivos.

- Integrar `fmt`, `validate`, `plan` y `apply` en un *workflow* gobernado.
- Definir criterios explícitos para mantener *jobs* en *hosted* o moverlos a *self-hosted*.
- Aplicar controles por entorno antes de cambios destructivos.
- Evitar anti-patrones frecuentes: `apply` sin revisión, permisos amplios y promoción sin trazabilidad.

## Resultado esperado al terminar.

- Un flujo reproducible con separación clara entre validación y ejecución de cambios.
- Una política de decisión para elegir runner por fase.
- Un checklist mínimo para aprobar cambios de infraestructura con menor riesgo.


## Mini glosario operativo de IaC.


- **Estado (`terraform.tfstate`)**: snapshot de la infraestructura real que Terraform usa para calcular diferencias.
- **Backend remoto**: ubicación compartida del estado (por ejemplo, bucket), con control de acceso y bloqueo de concurrencia.
- **Drift**: desviación entre lo declarado en código y lo que existe en cloud por cambios manuales.
- **Idempotencia**: aplicar varias veces el mismo código produce el mismo resultado final.
- **Plan binario (`-out=tfplan.binary`)**: artefacto inmutable del cambio aprobado que luego debe consumirse en `apply`.

Este vocabulario ayuda a revisar PR de infraestructura con criterios técnicos y no solo por intuición.

## Modelo operativo: de validación a cambio controlado.

Para *IaC*, este es el orden operativo recomendado del curso:

1. escribir o ajustar código *Terraform*;
2. ejecutar `fmt`, `validate` y `plan`;
3. revisar impacto por entorno;
4. aprobar explícitamente en entornos sensibles;
5. ejecutar `apply` con trazabilidad.

Separar `plan` y `apply` no es burocracia: reduce riesgo, mejora revisión y evita cambios de infraestructura sin contexto.


## Estrategia de *runners* por fase.

En la mayoría de equipos, una estrategia sana es:

- `fmt`, `validate`, `plan` en *hosted runners*;
- `apply` en *hosted* si no hay dependencias de red privada;
- `apply` en *self-hosted* solo cuando se requiere acceso a recursos internos, segmentación o controles regulatorios.

Si se usa *self-hosted*, debe ser preferentemente efímero, con limpieza post-*job*, parcheo continuo y credenciales de vida corta.


## Ejemplo alineado al repositorio: separación real de `plan` y `apply` con *OIDC*.

```yaml
name: terraform-plan-apply

on:
  workflow_dispatch:
    inputs:
      target_environment:
        type: choice
        options: [test, prod]
      run_apply:
        type: choice
        options: ["false", "true"]

permissions:
  contents: read
  id-token: write

concurrency:
  group: terraform-${{ inputs.target_environment }}
  cancel-in-progress: false

jobs:
  plan:
    runs-on: ubuntu-latest
    environment: ${{ inputs.target_environment }}
    outputs:
      plan_artifact: tfplan-${{ inputs.target_environment }}
    steps:
      - uses: actions/checkout@v6
      - uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ vars.GCP_WORKLOAD_IDENTITY_PROVIDER }}
          service_account: ${{ vars.GCP_SERVICE_ACCOUNT }}
      - uses: hashicorp/setup-terraform@v4
      - run: terraform init -backend-config="bucket=${{ vars.TF_STATE_BUCKET }}"
      - run: terraform fmt -check -recursive
      - run: terraform validate
      - run: terraform plan -out=tfplan.binary
      - run: terraform show -no-color tfplan.binary > tfplan.txt
      - uses: actions/upload-artifact@v4
        with:
          name: tfplan-${{ inputs.target_environment }}
          path: |
            tfplan.binary
            tfplan.txt

  apply:
    needs: plan
    if: ${{ inputs.run_apply == 'true' }}
    runs-on: ubuntu-latest
    environment: ${{ inputs.target_environment }}
    steps:
      - uses: actions/checkout@v6
      - uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ vars.GCP_WORKLOAD_IDENTITY_PROVIDER }}
          service_account: ${{ vars.GCP_SERVICE_ACCOUNT }}
      - uses: hashicorp/setup-terraform@v4
      - run: terraform init -backend-config="bucket=${{ vars.TF_STATE_BUCKET }}"
      - uses: actions/download-artifact@v4
        with:
          name: tfplan-${{ inputs.target_environment }}
      - run: terraform apply -auto-approve tfplan.binary
```

Este patrón minimiza riesgo porque el `apply` usa el mismo plan aprobado, conserva trazabilidad (`tfplan.txt`) y permite usar aprobaciones por entorno antes de ejecutar cambios destructivos.


In [ ]:
from dataclasses import dataclass

@dataclass
class DecisionInput:
    entorno: str
    run_apply: bool
    red_privada: bool
    regulado: bool

def estrategia_runner(entorno: str, red_privada: bool, regulado: bool) -> str:
    if red_privada or regulado:
        return "self-hosted-efimero"
    if entorno == "prod":
        return "hosted-con-aprobacion"
    return "hosted"

def politica_apply(entorno: str, run_apply: bool) -> str:
    if not run_apply:
        return "Solo PLAN (apply deshabilitado por input)"
    if entorno == "prod":
        return "APPLY condicionado a aprobacion del entorno"
    return "APPLY habilitado en entorno no productivo"

def evaluar(d: DecisionInput) -> dict:
    runner = estrategia_runner(d.entorno, d.red_privada, d.regulado)
    apply_policy = politica_apply(d.entorno, d.run_apply)
    riesgo = "alto" if d.entorno == "prod" else "medio"
    if d.red_privada or d.regulado:
        riesgo = "alto"
    return {
        "entorno": d.entorno,
        "runner_recomendado": runner,
        "apply": apply_policy,
        "riesgo_operativo": riesgo,
    }

escenarios = [
    DecisionInput("test", False, False, False),
    DecisionInput("test", True, True, False),
    DecisionInput("prod", False, False, False),
    DecisionInput("prod", True, True, True),
]

print("Decision rapida de pipeline de infraestructura")
print("=" * 72)
for i, s in enumerate(escenarios, start=1):
    r = evaluar(s)
    print(f"\nEscenario {i}")
    print(f"  entorno            : {r['entorno']}")
    print(f"  runner recomendado : {r['runner_recomendado']}")
    print(f"  politica apply     : {r['apply']}")
    print(f"  riesgo operativo   : {r['riesgo_operativo']}")

## Checklist mínima antes de ejecutar `apply`.

Usa esta lista como puerta de salida del `plan`:

- `terraform fmt -check` y `terraform validate` en verde.
- `tfplan.txt` revisado por otra persona en cambios sensibles.
- Entorno correcto seleccionado (evitar promoción accidental).
- Credenciales efímeras vía *OIDC* activas; sin llaves estáticas en el flujo.
- Estrategia de runner justificada (hosted por defecto; self-hosted solo por necesidad real).
- Ventana de cambio y plan de rollback definidos.

Si una condición falla, se vuelve a `plan` y no se ejecuta `apply`.

## Ejercicio corto: rollback de infraestructura.


Objetivo: practicar recuperación controlada después de un cambio que degradó la conectividad o el hardening esperado.

1. Genera y guarda el plan actual.

```bash
terraform plan -out=tfplan.binary
terraform show -no-color tfplan.binary > tfplan.txt
```

2. Simula un cambio de riesgo (por ejemplo, ampliar `source_ranges` de una regla).

3. Genera un nuevo plan y compáralo con el plan aprobado.

4. Ejecuta rollback aplicando el artefacto aprobado anterior (si sigue siendo válido para el estado actual) o revirtiendo en Git y regenerando plan.

```bash
# Opción A: aplicar plan aprobado previamente
terraform apply tfplan.binary

# Opción B: revertir commit, regenerar plan y aplicar
git revert <sha-del-cambio>
terraform plan -out=tfplan-rollback.binary
terraform apply tfplan-rollback.binary
```

5. Verifica recuperación: conectividad esperada, reglas de firewall correctas y ausencia de drift crítico.


## Rúbrica mínima para revisión de PR de IaC.


- **Correctitud**: `fmt` y `validate` pasan sin excepciones.
- **Impacto**: `tfplan.txt` está adjunto y revisado en cambios sensibles.
- **Seguridad**: IAM en mínimo privilegio; sin aperturas de red amplias por defecto.
- **Gobierno**: entorno objetivo y aprobación explícita cuando aplica.
- **Operación**: rollback definido y comprobable.
- **Trazabilidad**: plan aplicado corresponde al plan revisado.

## Tensión entre mínimo privilegio y permisos del runner de Terraform.

Este es un anti-patrón que aparece en la práctica y que la rúbrica anterior ("IAM en mínimo privilegio") no captura por sí sola.

### El problema

Cuando Terraform gestiona recursos de tipo `google_[servicio]_[recurso]_iam_member`, necesita llamar a `getIamPolicy` sobre ese recurso en cada `plan` para leer el estado actual de los bindings. Ese permiso **no siempre está incluido en el rol de uso del recurso**.

Ejemplo concreto con Artifact Registry:

| Operación | Permiso requerido | Incluido en `writer` | Incluido en `repoAdmin` (repo) | Incluido en `admin` (proyecto) |
|---|---|---|---|---|
| `docker push` | `artifactregistry.repositories.uploadArtifacts` | ✓ | ✓ | ✓ |
| Leer IAM del repo | `artifactregistry.repositories.getIamPolicy` | ✗ | ✗ | ✓ |
| Modificar IAM del repo | `artifactregistry.repositories.setIamPolicy` | ✗ | ✗ | ✓ |

El pipeline de CI/CD solo necesita `writer`. Pero Terraform, para gestionar los bindings IAM de ese mismo repositorio, necesita `roles/artifactregistry.admin` a **nivel de proyecto** — un rol mucho más amplio.

### Por qué no se detectó antes

Los `terraform apply` locales corren con la cuenta personal del operador (que tiene `Owner`). El error solo aparece cuando el **workflow de CI/CD** ejecuta Terraform bajo el service account con mínimo privilegio. Es un caso clásico de "funciona en local, falla en CI".

### La solución arquitectónica correcta

Separar el service account del runner de Terraform del service account del pipeline de despliegue:

```
terraform-runner@...   ← ejecuta terraform apply; permisos amplios de gestión
cicd-deployer@...      ← ejecuta docker push y gcloud run deploy; permisos mínimos
```

Cada uno tiene su propio WIF binding restringido al workflow que lo necesita:

```hcl
# Solo el workflow de infra puede usar terraform-runner
member = "principalSet://.../attribute.job_workflow_ref/refs/heads/main:infra.yaml"

# Solo el workflow de deploy puede usar cicd-deployer
member = "principalSet://.../attribute.environment/prod"
```

### Pragmatismo del curso

Este repositorio usa un único SA (`cicd-deployer`) para ambos roles, con `roles/artifactregistry.admin` añadido a nivel de proyecto para desbloquear el runner de Terraform. Es una simplificación válida para un proyecto de aprendizaje, pero en un entorno productivo real se debe aplicar la separación de cuentas.

El trade-off está documentado en `iam.tf` junto a la declaración del rol.

## Puente con el resto del bloque.

Este flujo conecta construcción de artefactos, gobierno de infraestructura y despliegue seguro en *Linux*:

- el mismo criterio de endurecimiento aplica a *build* y a *IaC*;
- los entornos protegidos definen quién promueve cambios;
- la operación en *Linux* hereda la misma trazabilidad del *pipeline*.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>